# EcoSort Waste Management Assistant
# Module 8 Summative Lab

## Overview

You are a data scientist at "EcoSort," a technology company that specializes in developing AI solutions for waste management. EcoSort has partnered with Metro City's waste management department to develop an intelligent waste management assistant that can help residents properly dispose of waste items so less time is spent sorting material at facilities.

This assistant needs to:

1. Identify waste materials from images uploaded by residents (CNN)
2. Classify waste items based on text descriptions provided by residents (RNN/Transformer)
3. Generate specific recycling instructions based on identified waste type and city policies (Generative Transformer with RAG)

Your task is to build this integrated system using the RealWaste dataset along with generated text data that simulates real-world waste management operations.

## Part 1: Dataset Exploration and Preparation

In this section, you will explore and prepare the datasets for your models.

### 1.1 Load and Explore the RealWaste Dataset

In [ ]:
# Import necessary libraries
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import image_dataset_from_directory, load_img, img_to_array
from tensorflow.keras.layers import TextVectorization, Embedding
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

from PIL import Image
from collections import Counter
import json
import re
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import pathlib



# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

In [ ]:
# Dataset path
DATASET_PATH = "./realwaste-main/RealWaste/"

# Get waste categories
categories = os.listdir(DATASET_PATH)

# Count images in each category
category_counts = {}

for category in categories:
    category_path = os.path.join(DATASET_PATH, category)

    if os.path.isdir(category_path):
        images = os.listdir(category_path)
        category_counts[category] = len([
            img for img in images
            if img.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

# Create DataFrame
category_df = pd.DataFrame(
    list(category_counts.items()),
    columns=["Category", "Image Count"]
)

print("Number of Categories:", len(category_df))
print("Total Images:", category_df["Image Count"].sum())

display(category_df)

In [ ]:
# - Distribution of waste categories

plt.figure(figsize=(12, 5))
sns.barplot(data=category_df, x="Category", y="Image Count")

plt.title("RealWaste Category Distribution")
plt.xlabel("Waste Category")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)

plt.show()

In [ ]:
# - Image characteristics (resolution, quality, background)
# image resolution distribution

# Collect image dimensions
image_info = []

for category in categories:
    category_path = os.path.join(DATASET_PATH, category)

    if not os.path.isdir(category_path):
        continue

    for filename in os.listdir(category_path):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            image_path = os.path.join(category_path, filename)

            with Image.open(image_path) as img:
                width, height = img.size
                image_info.append([width, height])

# Create DataFrame
image_df = pd.DataFrame(image_info, columns=["Width", "Height"])

# Display image statistics
display(image_df.describe())

# Visualize image dimensions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(image_df["Width"], bins=30, ax=axes[0])
axes[0].set_title("Image Width Distribution")

sns.histplot(image_df["Height"], bins=30, ax=axes[1])
axes[1].set_title("Image Height Distribution")

plt.show()

In [ ]:
# sample images

# Display 3 sample images from each category
plt.figure(figsize=(12, 3 * len(category_df)))

for i, category in enumerate(category_df["Category"]):
    category_path = os.path.join(DATASET_PATH, category)

    images = [
        img for img in os.listdir(category_path)
        if img.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    sample_images = random.sample(images, min(3, len(images)))

    for j, filename in enumerate(sample_images):
        image_path = os.path.join(category_path, filename)

        img = Image.open(image_path)

        plt.subplot(len(category_df), 3, i * 3 + j + 1)
        plt.imshow(img)
        plt.title(category)
        plt.axis("off")

plt.tight_layout()
plt.show()

### 1.2 Explore Text Datasets

In [ ]:
# TODO: Load and explore the waste description text data
# - Load waste_descriptions.csv
text_df = pd.read_csv("waste_descriptions.csv")

text_df.head(10)

In [ ]:
# - Analyze vocabulary and structure

# word count for each description
text_df["word_count"] = text_df["description"].fillna("").str.split().str.len()

print("Description Length Statistics:")
display(text_df["word_count"].describe())

# visualize description lengths
plt.figure(figsize=(10, 5))
sns.histplot(text_df["word_count"])

plt.title("Distribution of Description Lengths")
plt.xlabel("Number of words")
plt.ylabel("Number of descriptions")
plt.show()


In [ ]:
# combine descriptions into one string
all_text = " ".join(text_df["description"].dropna()).lower()

# split text into individual words
words = all_text.split()

# count unique words
vocabulary = set(words)

print("Total Words:", len(words))
print("Unique Words:", len(vocabulary))

# find 20 most common words
word_counts = Counter(words)

print("\n20 most common words:")
display(pd.DataFrame(
    word_counts.most_common(20),
    columns=["Word", "Count"]
))

In [ ]:
# - Understand the distribution of categories
# Count descriptions in each category
category_counts = text_df["category"].value_counts()

# Display category counts
print("Waste Category Distribution:")
display(category_counts)

# Plot category distribution
plt.figure(figsize=(12, 6))
sns.barplot(x=category_counts.index, y=category_counts.values)

plt.title("Waste Description Category Distribution")
plt.xlabel("Waste Category")
plt.ylabel("Number of Descriptions")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# TODO: Load and explore the waste policy documents
# - Load waste_policy_documents.csv
with open("waste_policy_documents.json", "r") as file:
    policies = json.load(file)

# convert to DF
policy_df = pd.DataFrame(policies)

# Display first 5 policies
policy_df.head()

In [ ]:
# - Understand document organization and language

# Document organization

print("\nColumn Names:")
print(policy_df.columns.tolist())

print("\nDataset Information:")
policy_df.info()

# Count policies by type
print("\nPolicy Types:")
display(policy_df["policy_type"].value_counts())

# Display categories covered
print("\nCategories Covered:")
display(policy_df["categories_covered"].explode().value_counts())

# Display jurisdictions
print("\nJurisdications:")
display(policy_df["jurisdiction"].explode().value_counts())

In [ ]:
# Document language

# Calculate word count for each policy
policy_df["word_count"] = policy_df["document_text"].str.split().str.len()

# Display word count statistics
print("Policy Document Length Statistics:")
display(policy_df["word_count"].describe())

# Plot document lengths
plt.figure(figsize=(10, 5))
sns.histplot(policy_df["word_count"])

plt.title("Waste Policy Document Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Number of Policies")
plt.show()

### 1.3 Create Data Pipelines

In [ ]:
# Run this code to setup the images properly into train, validation, and test sets
# Set your data directory path - update this with your actual path
data_dir = pathlib.Path('./realwaste-main/RealWaste/')

# Parameters
BATCH_SIZE = 32
IMG_HEIGHT = 224
IMG_WIDTH = 224

# Calculate the total number of classes automatically from the directory structure
num_classes = len([item for item in data_dir.glob('*') if item.is_dir()])
print(f"Number of classes: {num_classes}")

# List all class folders
class_names = sorted([item.name for item in data_dir.glob('*') if item.is_dir()])
print(f"Class names: {class_names}")

# Count all images
image_count = len(list(data_dir.glob('*/*.jpg'))) + len(list(data_dir.glob('*/*.png')))
print(f"Total images found: {image_count}")

# Create a dataset using tf.keras.utils.image_dataset_from_directory
# This will automatically split the data into training and validation sets
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,  # 20% for validation
    subset="training",
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='categorical',  # For one-hot encoded labels
    shuffle=True
)

validation_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,  # 20% for validation
    subset="validation",
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='categorical',  # For one-hot encoded labels
    shuffle=False
)

# Create a separate test dataset by taking part of the validation set
# First, let's get the number of batches in the validation set
val_batches = tf.data.experimental.cardinality(validation_ds)
test_dataset = validation_ds.take(val_batches // 2)
validation_ds = validation_ds.skip(val_batches // 2)

print(f"Number of training batches: {tf.data.experimental.cardinality(train_ds)}")
print(f"Number of validation batches: {tf.data.experimental.cardinality(validation_ds)}")
print(f"Number of test batches: {tf.data.experimental.cardinality(test_dataset)}")

# Configure dataset for performance
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
validation_ds = validation_ds.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

In [ ]:
# TODO: Create a text preprocessing pipeline
# - Text cleaning

# Load text dataset if not already loaded
text_df = pd.read_csv("waste_descriptions.csv")

# Remove rows with missing descriptions or categories
text_df = text_df.dropna(subset=["description", "category"]).copy()

# Function to clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.strip()

# Apply cleaning
text_df["clean_description"] = text_df["description"].apply(clean_text)

# Display cleaned descriptions
display(text_df[["description", "clean_description"]].head())

In [ ]:
# Encode waste cetegories

# Convert category names to numerical labels
label_encoder = LabelEncoder()

text_df["label"] = label_encoder.fit_transform(text_df["category"])

# Display category mappings
for i, category in enumerate(label_encoder.classes_):
    print(f"{i}: {category}")

# Number of classes
num_text_classes = len(label_encoder.classes_)

In [ ]:
# - Split data into train and test
X = text_df["clean_description"]
y = text_df["label"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
# - Tokenization

# Set vocabulary size and sequence length
MAX_TOKENS = 5000
MAX_LENGTH = 30

# Create text vectorization layer
vectorizer = TextVectorization(max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=MAX_LENGTH
)

# Learn vocabulary from training data only
vectorizer.adapt(X_train.to_numpy())

# Convert text into numerical sequences
X_train_seq = vectorizer(X_train.to_numpy())
X_test_seq = vectorizer(X_test.to_numpy())

print("Training Sequence Shape:", X_train_seq.shape)
print("Testing Sequence Shape:", X_test_seq.shape)

# Display an example
print("\nOriginal Description:")
print(X_train.iloc[0])

print("\nTokenized Sequence:")
print(X_train_seq[0].numpy())


In [ ]:
# - Create embeddings/features
# Define embedding dimensions
EMBEDDING_DIM = 64

# Create embedding layer
embedding_layer = Embedding(
    input_dim=MAX_TOKENS,
    output_dim=EMBEDDING_DIM,
    mask_zero=True
)

# Generate embeddings for a sample
sample_embedding = embedding_layer(X_train_seq[:1])

print("Embedding Shape:", sample_embedding.shape)

In [ ]:
# TODO: Prepare documents for RAG
# - Document preprocessing

# Handle missing values
policy_df["document_text"] = policy_df["document_text"].fillna("")
policy_df["policy_type"] = policy_df["policy_type"].fillna("")

# Combine relevant information into one text field
policy_df["processed_text"] = (
    policy_df["policy_type"] + ". " +
    policy_df["categories_covered"].apply(lambda x: ", ".join(x)) + ". " +
    policy_df["document_text"]
)

# Remove extra whitespace
policy_df["processed_text"] = (
    policy_df["processed_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Display example
print(policy_df["processed_text"].iloc[0])

In [ ]:
# - Create embeddings for retrieval

# Load pretrained embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings for policy documents
document_embeddings = embedding_model.encode(
    policy_df["processed_text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Document Embeddings Shape:", document_embeddings.shape)

## Part 2: Waste Material Classification with CNN

In this section, you will build a CNN model to classify waste materials from images.

### 2.1 Preprocess Images

In [ ]:
# TODO: Implement image preprocessing
# - Apply the preprocessing pipeline created earlier

# Normalize pixel values from 0-255 to 0-1
def normalize_images(images, labels):
    images = tf.cast(images, tf.float32) / 255.0
    return images, labels

# Apply normalization to all datasets
train_ds = train_ds.map(normalize_images)
validation_ds = validation_ds.map(normalize_images)
test_dataset = test_dataset.map(normalize_images)

# Check image shapes and pixel values
for images, labels in train_ds.take(1):
    print("Image Shape:", images.shape)
    print("Label Shape:", labels.shape)
    print("Minimum Pixel Value:", tf.reduce_min(images).numpy())
    print("Maximum Pixel Value:", tf.reduce_max(images).numpy())

In [ ]:
# Create image augmentation pipeline
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

# Display original and augmented images
for images, labels in train_ds.take(1):
    plt.figure(figsize=(10, 5))

    for i in range(5):
        augmented_image = data_augmentation(
            images[0:1], training=True
        )
        plt.subplot(1, 5, i + 1)
        plt.imshow(augmented_image[0].numpy())
        plt.axis("off")

    plt.suptitle("Sample Augmented Images")
    plt.show()

### 2.2 Implement CNN Model with Transfer Learning

In [ ]:
# TODO: Select an appropriate base model and implement transfer learning
# - Choose from MobileNet, EfficientNet, etc.
# - Add custom classification layers for the 9 waste categories
# - Configure loss function and metrics

# Load pretrained MobileNetV2 without its classification layer
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained layers
base_model.trainable = False

print("MobileNetV2 loaded successfully")

In [ ]:
# Build CNN model using transfer learning
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    # Image augmentation
    data_augmentation,

    # Convert normalized pixels from [0, 1] to [-1, 1]
    layers.Rescaling(2.0, offset=-1.0),

    # Pretrained MobileNetV2
    base_model,

    # Custom classification layers
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

# Display model architecture
model.summary()

In [ ]:
# Compile CNN model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("CNN model compiled successfully")

In [ ]:
# Display model summary
model.summary()

# Check number of trainable parameters
print("Trainable Parameters:",
      sum(tf.size(w).numpy() for w in model.trainable_weights))

print("Number of Waste Categories:", num_classes)

### 2.3 Train and Evaluate the Model

In [ ]:
# TODO: Train the CNN model
# - Use appropriate batch size and epochs
# - Implement regularization to prevent overfitting
# - Monitor training and validation metrics

# Configure training callbacks
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    ModelCheckpoint(
        "best_waste_cnn.keras",
        monitor="val_loss",
        save_best_only=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
]

In [ ]:
# Train CNN model
history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=20,
    callbacks=callbacks
)

In [ ]:
# Plot training and validation accuracy
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("CNN Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

# Plot training and validation loss
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("CNN Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Evaluate model performance
# - Calculate accuracy on test set
# - Generate confusion matrix
# - Analyze error patterns

# Evaluate model on test data
test_loss, test_accuracy = model.evaluate(test_dataset)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
# Get actual labels and model predictions
y_true = []
y_pred = []

for images, labels in test_dataset:
    predictions = model.predict(images, verbose=0)

    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(predictions, axis=1))

# Convert to NumPy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Total Test Images:", len(y_true))
print("Correct Predictions:", np.sum(y_true == y_pred))
print("Incorrect Predictions:", np.sum(y_true != y_pred))

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=range(len(class_names))
)

# Plot confusion matrix
plt.figure(figsize=(12, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("CNN Confusion Matrix")
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Generate classification report
print("CNN Classification Report:")

print(classification_report(
    y_true,
    y_pred,
    labels=range(len(class_names)),
    target_names=class_names,
    zero_division=0
))

# Calculate errors by category
errors = cm.sum(axis=1) - np.diag(cm)

error_df = pd.DataFrame({
    "Category": class_names,
    "Total Images": cm.sum(axis=1),
    "Incorrect Predictions": errors,
    "Error Rate (%)": np.divide(
        errors,
        cm.sum(axis=1),
        out=np.zeros(len(errors), dtype=float),
        where=cm.sum(axis=1) != 0
    ) * 100
})

# Sort by error rate
error_df = error_df.sort_values(
    "Error Rate (%)",
    ascending=False
)

print("\nClassification Errors by Category:")
display(error_df)

### 2.4 Fine-tune the Model

In [ ]:
# TODO: Tune model parameters to improve performance
# - Adjust learning rate
# - Add regularization, dropout
# - Modify architecture if needed

# Allow fine-tuning of MobileNetV2
base_model.trainable = True

# Freeze all layers except the last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep BatchNormalization layers frozen
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

print("Trainable layers:",
      sum(layer.trainable for layer in base_model.layers))

In [ ]:
# Add L2 regularization to the Dense layer
dense_layer = next(
    layer for layer in model.layers
    if isinstance(layer, tf.keras.layers.Dense)
    and layer.units == 128
)

dense_layer.kernel_regularizer = regularizers.l2(0.001)

# Recompile with a smaller learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# Fine-tune the model
fine_tune_history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=10,
    callbacks=callbacks
)

In [ ]:
# Evaluate fine-tuned model
tuned_loss, tuned_accuracy = model.evaluate(test_dataset)

print(f"Original Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Tuned Test Accuracy: {tuned_accuracy * 100:.2f}%")
print(f"Accuracy Improvement: {(tuned_accuracy - test_accuracy) * 100:.2f} percentage points")

# Plot fine-tuning accuracy and loss
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(fine_tune_history.history["accuracy"], label="Training")
plt.plot(fine_tune_history.history["val_accuracy"], label="Validation")
plt.title("Fine-Tuned CNN Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(fine_tune_history.history["loss"], label="Training")
plt.plot(fine_tune_history.history["val_loss"], label="Validation")
plt.title("Fine-Tuned CNN Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## Part 3: Waste Description Classification

In this section, you will build a text classification model to categorize waste based on descriptions.

### 3.1 Preprocess Text Data

In [ ]:
# TODO: Implement text preprocessing
# - Apply the text preprocessing pipeline created earlier

# Convert descriptions into numerical sequences
X_train_seq = vectorizer(X_train.to_numpy())
X_test_seq = vectorizer(X_test.to_numpy())

# Convert labels to NumPy arrays
y_train_array = y_train.to_numpy()
y_test_array = y_test.to_numpy()

# Display dataset shapes
print("Training Features:", X_train_seq.shape)
print("Testing Features:", X_test_seq.shape)
print("Training Labels:", y_train_array.shape)
print("Testing Labels:", y_test_array.shape)

In [ ]:
# Display an example description
print("Original Description:")
print(X_train.iloc[0])

print("\nTokenized Sequence:")
print(X_train_seq[0].numpy())

print("\nActual Category:")
print(label_encoder.inverse_transform([y_train_array[0]])[0])

print("\nNumber of Categories:", num_text_classes)

### 3.2 Implement Text Classification Model

In [ ]:
# TODO: Choose and implement a text classification model
# Option A: Traditional ML model (Naive Bayes, Random Forest, etc.)
# Option B: Fine-tune a transformer-based model (BERT, DistilBERT, etc.)

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=5000)

# Convert text into numerical features
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Display feature shapes
print("Training Features:", X_train_tfidf.shape)
print("Testing Features:", X_test_tfidf.shape)

In [ ]:

# Create Naive Bayes model
text_model = MultinomialNB()

# Train model
text_model.fit(X_train_tfidf, y_train)

print("Naive Bayes model trained successfully!")

In [ ]:
# Predict waste categories
y_pred_text = text_model.predict(X_test_tfidf)

# Display sample predictions
results_df = pd.DataFrame({
    "Description": X_test.values,
    "Actual": label_encoder.inverse_transform(y_test.values),
    "Predicted": label_encoder.inverse_transform(y_pred_text)
})

display(results_df.head(10))

In [ ]:
# Test a new waste description
new_description = ["empty plastic water bottle"]

# Apply the same text cleaning
cleaned_text = [clean_text(text) for text in new_description]

# Convert to TF-IDF features
new_features = tfidf.transform(cleaned_text)

# Predict category
prediction = text_model.predict(new_features)

# Convert prediction to category name
predicted_category = label_encoder.inverse_transform(prediction)

print("Description:", new_description[0])
print("Predicted Category:", predicted_category[0])

### 3.3 Train and Evaluate the Model

In [ ]:
# TODO: Train the text classification model
# - Use appropriate training parameters
# - Monitor training progress

# Train the Naive Bayes model
text_model.fit(X_train_tfidf, y_train)

# Calculate training accuracy
train_predictions = text_model.predict(X_train_tfidf)
train_accuracy = accuracy_score(y_train, train_predictions)

print(f"Training Accuracy: {train_accuracy * 100:.2f}%")

In [ ]:
# Predict waste categories on test data
y_pred_text = text_model.predict(X_test_tfidf)

# Calculate test accuracy
test_accuracy_text = accuracy_score(y_test, y_pred_text)

print(f"Training Accuracy: {train_accuracy * 100:.2f}%")
print(f"Test Accuracy: {test_accuracy_text * 100:.2f}%")

In [ ]:
# Generate classification report
print("Text Classification Report:")

print(classification_report(
    y_test,
    y_pred_text,
    target_names=label_encoder.classes_,
    zero_division=0
))

In [ ]:
# TODO: Evaluate model performance
# - Calculate accuracy on test set
# - Generate confusion matrix
# - Analyze error patterns

# Predict categories using the test data
y_pred_text = text_model.predict(X_test_tfidf)

# Calculate accuracy
test_accuracy_text = accuracy_score(y_test, y_pred_text)

print(f"Test Accuracy: {test_accuracy_text * 100:.2f}%")

In [ ]:
# Generate confusion matrix
cm_text = confusion_matrix(y_test, y_pred_text)

# Plot confusion matrix
plt.figure(figsize=(12, 8))

sns.heatmap(
    cm_text,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.title("Text Classification Confusion Matrix")
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Generate classification report
print("Text Classification Report:")

print(classification_report(
    y_test,
    y_pred_text,
    labels=range(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
))

In [ ]:
# Calculate errors for each category
cm_text = confusion_matrix(
    y_test,
    y_pred_text,
    labels=range(len(label_encoder.classes_))
)

errors = cm_text.sum(axis=1) - np.diag(cm_text)

# Create error analysis DataFrame
error_df_text = pd.DataFrame({
    "Category": label_encoder.classes_,
    "Total Descriptions": cm_text.sum(axis=1),
    "Incorrect Predictions": errors
})

# Calculate error rate
error_df_text["Error Rate (%)"] = (
    error_df_text["Incorrect Predictions"] /
    error_df_text["Total Descriptions"].replace(0, np.nan) * 100
).fillna(0).round(2)

# Sort by highest error rate
error_df_text = error_df_text.sort_values(
    "Error Rate (%)",
    ascending=False
)

print("Classification Errors by Category:")
display(error_df_text)

In [ ]:
# Create DataFrame of predictions
results_df = pd.DataFrame({
    "Description": X_test.values,
    "Actual": label_encoder.inverse_transform(y_test.to_numpy()),
    "Predicted": label_encoder.inverse_transform(y_pred_text)
})

# Filter incorrect predictions
incorrect_df = results_df[
    results_df["Actual"] != results_df["Predicted"]
]

print("Total Incorrect Predictions:", len(incorrect_df))

display(incorrect_df.head(20))

### 3.4 Create Classification Function

In [ ]:
# TODO: Create a function that takes a text description and returns the predicted waste category

def classify_waste_description(description):
    """
    Classifies a waste description into an appropriate category.

    Args:
        description (str): Text description of waste item

    Returns:
        str: Predicted waste category
    """
    # Your code here
    
    # Clean the description
    cleaned_text = clean_text(description)

    # Convert text into TF-IDF features
    features = tfidf.transform([cleaned_text])

    # Predict waste category
    prediction = text_model.predict(features)

    # Convert numerical prediction to category name
    category = label_encoder.inverse_transform(prediction)[0]

    return category

In [ ]:
# Test with sample waste descriptions
test_descriptions = [
    "empty plastic water bottle",
    "broken glass jar",
    "old cardboard shipping box",
    "banana peel and leftover vegetables",
    "used paper newspaper",
    "old cotton shirt",
    "empty aluminum soda can"
]

# Classify each description
for description in test_descriptions:
    category = classify_waste_description(description)

    print(f"Description: {description}")
    print(f"Predicted Category: {category}")
    print("-" * 50)

## Part 4: Recycling Instruction Generation with RAG

In this section, you will implement a Retrieval-Augmented Generation (RAG) system to generate recycling instructions.

### 4.1 Preprocess Documents for Retrieval

In [ ]:
# TODO: Prepare documents for retrieval
# - Process policy documents and disposal instructions
# - Create embeddings for efficient retrieval

# Your code here

### 4.2 Implement RAG-based System

In [ ]:
# TODO: Select a pre-trained language model and implement RAG
# - Choose an appropriate language model
# - Create a retrieval mechanism

# Prepare policy documents for retrieval
policy_docs = policy_df.copy()

# Combine policy information into searchable text
policy_docs["search_text"] = (
    policy_docs["policy_type"] + ". " +
    policy_docs["categories_covered"].apply(lambda x: ", ".join(x)) + ". " +
    policy_docs["document_text"]
)

# Display sample
display(policy_docs[["policy_type", "search_text"]].head())

In [ ]:
# Prepare disposal instructions
disposal_docs = text_df.copy()

# Remove rows with missing instructions
disposal_docs = disposal_docs.dropna(
    subset=["description", "category", "disposal_instruction"]
)

# Combine descriptions, categories, and instructions
disposal_docs["search_text"] = (
    disposal_docs["category"] + ". " +
    disposal_docs["description"] + ". " +
    disposal_docs["disposal_instruction"]
)

# Display sample
display(disposal_docs[["category", "search_text"]].head())

In [ ]:
# Create a combined document collection
policy_records = pd.DataFrame({
    "source": "Policy",
    "category": policy_docs["categories_covered"].apply(
        lambda x: ", ".join(x)
    ),
    "text": policy_docs["search_text"]
})

disposal_records = pd.DataFrame({
    "source": "Disposal Instruction",
    "category": disposal_docs["category"],
    "text": disposal_docs["search_text"]
})

# Combine both sources
rag_documents = pd.concat(
    [policy_records, disposal_records],
    ignore_index=True
)

print("Total RAG Documents:", len(rag_documents))
print("\nDocuments by Source:")
print(rag_documents["source"].value_counts())

display(rag_documents.head())

In [ ]:
# Generate embeddings for all RAG documents
rag_embeddings = embedding_model.encode(
    rag_documents["text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Total Documents:", len(rag_documents))
print("Embedding Shape:", rag_embeddings.shape)

### 4.3 Adjust and Evaluate the System

In [ ]:
# TODO: Train the RAG-based system
# - Adjust sampling methods/parameters

# RAG retrieval parameters
TOP_K = 3
MIN_SIMILARITY = 0.30

# Text generation parameters
MAX_NEW_TOKENS = 150
TEMPERATURE = 0.3
TOP_P = 0.9

print("RAG Parameters:")
print("Top K:", TOP_K)
print("Minimum Similarity:", MIN_SIMILARITY)
print("Max New Tokens:", MAX_NEW_TOKENS)
print("Temperature:", TEMPERATURE)
print("Top P:", TOP_P)

In [ ]:
def retrieve_documents(query, top_k=3, min_similarity=0.30):

    # Convert question into embedding
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    # Calculate similarity scores
    similarities = cosine_similarity(
        query_embedding,
        rag_embeddings
    )[0]

    # Find most relevant documents
    top_indices = np.argsort(similarities)[::-1][:top_k]

    # Return documents above similarity threshold
    results = rag_documents.iloc[top_indices].copy()
    results["similarity"] = similarities[top_indices]

    results = results[
        results["similarity"] >= min_similarity
    ]

    return results

In [ ]:
# Example waste description
query = "How should I recycle a glass bottle?"

# Test retrieval
results = retrieve_documents(
    query,
    top_k=TOP_K,
    min_similarity=MIN_SIMILARITY
)

# Display results
display(results[["source", "category", "similarity", "text"]])

In [ ]:

# Load pretrained text generation model
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

In [ ]:
# Generate recycling instructions from a waste description

def generate_instructions_from_description(description):
    """
    Generates recycling instructions from a waste description.

    Args:
        description (str): Description of waste item

    Returns:
        str: Generated recycling instructions
    """

    # Retrieve relevant documents
    documents = retrieve_documents(
        description,
        top_k=TOP_K,
        min_similarity=MIN_SIMILARITY
    )

    # Check whether documents were found
    if documents.empty:
        return "No relevant recycling policy found."

    # Prefer official policy documents
    policy_matches = documents[
        documents["source"] == "Policy"
    ]

    if not policy_matches.empty:
        documents = policy_matches

    # Combine retrieved documents
    context = "\n".join(documents["text"].tolist())

    # Create prompt
    prompt = f"""
    Provide brief recycling instructions using only the policy context.
    If the context does not answer the question, say so.

    Context:
    {context[:1500]}

    Waste Item: {description}

    Recycling Instructions:
    """

    # Generate instructions
    response = generator(
        prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P
    )

    return response[0]["generated_text"]

In [ ]:
# Sample test cases
test_cases = {
    "empty glass bottle": "Glass",
    "old cotton shirt": "Textile Trash",
    "plastic water bottle": "Plastic",
    "banana peel": "Food Organics"
}

correct = 0

for description, expected_category in test_cases.items():

    documents = retrieve_documents(description, top_k=3)

    retrieved_categories = documents["category"].tolist()

    # Check whether the expected category was retrieved
    match = any(
        expected_category in category.split(", ")
        for category in retrieved_categories
    )

    correct += int(match)

    print(f"{description}: {'Correct' if match else 'Incorrect'}")

retrieval_accuracy = correct / len(test_cases)

print(f"\nRetrieval Accuracy: {retrieval_accuracy * 100:.2f}%")

In [ ]:
# Evaluate generated instructions across waste categories

test_items = {
    "Glass": "empty glass bottle",
    "Plastic": "plastic water bottle",
    "Paper": "old newspaper",
    "Cardboard": "cardboard shipping box",
    "Metal": "empty aluminum soda can",
    "Food Organics": "banana peel",
    "Textile Trash": "old cotton shirt",
    "Vegetation": "fallen tree leaves",
    "Miscellaneous Trash": "broken ceramic plate"
}

# Generate instructions for each category
evaluation_results = []

for expected_category, description in test_items.items():

    instructions = generate_instructions_from_description(
        description
    )

    evaluation_results.append({
        "Waste Category": expected_category,
        "Description": description,
        "Generated Instructions": instructions
    })

# Convert results to DataFrame
rag_eval_df = pd.DataFrame(evaluation_results)

display(rag_eval_df)

In [ ]:
relevance_scores = []

for _, row in rag_eval_df.iterrows():

    # Generate embeddings
    embeddings = embedding_model.encode([
        row["Description"],
        row["Generated Instructions"]
    ])

    # Calculate similarity
    similarity = cosine_similarity(
        [embeddings[0]],
        [embeddings[1]]
    )[0][0]

    relevance_scores.append(similarity)

# Add relevance scores
rag_eval_df["Relevance Score"] = relevance_scores

display(rag_eval_df[[
    "Waste Category",
    "Description",
    "Relevance Score"
]])

In [ ]:
# Create evaluation columns for manual review
rag_eval_df["Accuracy Score"] = np.nan
rag_eval_df["Comments"] = ""

# Display instructions for review
for index, row in rag_eval_df.iterrows():

    print(f"\nCategory: {row['Waste Category']}")
    print(f"Description: {row['Description']}")
    print(f"Generated Instructions: {row['Generated Instructions']}")

    print("\nRelevant Documents:")

    documents = retrieve_documents(
        row["Description"],
        top_k=3
    )

    for _, doc in documents.iterrows():
        print(f"- {doc['source']}: {doc['text'][:500]}")

    print("-" * 70)

In [ ]:
# Display relevance summary
print("RAG Evaluation Summary")
print("-" * 40)

print(f"Total Test Cases: {len(rag_eval_df)}")

print(
    f"Average Relevance Score: "
    f"{rag_eval_df['Relevance Score'].mean():.3f}"
)

# Show all results
display(rag_eval_df)

### 4.4 Create Instruction Generation Function

In [ ]:
# TODO: Create a function that takes a waste category and generates recycling instructions

def generate_recycling_instructions(waste_category):
    """
    Generates detailed recycling instructions for a given waste category.

    Args:
        waste_category (str): Waste category

    Returns:
        str: Detailed recycling instructions
        list: Relevant policy documents
    """
    # Your code here
    
    # Retrieve relevant documents for the waste category
    documents = retrieve_documents(
        waste_category,
        top_k=TOP_K,
        min_similarity=MIN_SIMILARITY
    )

    # Check if any documents were found
    if documents.empty:
        return "No relevant recycling policies found.", []

    # Keep official policy documents when available
    policy_documents = documents[
        documents["source"] == "Policy"
    ]

    if not policy_documents.empty:
        documents = policy_documents

    # Convert retrieved documents into a list
    relevant_policies = documents["text"].tolist()

    # Combine documents into context
    context = "\n".join(relevant_policies)

    # Create prompt for the language model
    prompt = f"""
    Generate clear recycling instructions for the waste category below.
    Use only the provided policy information.
    Do not invent recycling rules.
    If the policies do not contain enough information, say so.

    Waste Category: {waste_category}

    Policy Information:
    {context[:1500]}

    Recycling Instructions:
    """

    # Generate recycling instructions
    response = generator(
        prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P
    )

    # Extract generated text
    instructions = response[0]["generated_text"]

    return instructions, relevant_policies

In [ ]:
# Test recycling instruction generation
waste_category = "Plastic"

instructions, policies = generate_recycling_instructions(
    waste_category
)

print("Waste Category:", waste_category)

print("\nGenerated Recycling Instructions:")
print(instructions)

print("\nRelevant Policy Documents:")

for i, policy in enumerate(policies, start=1):
    print(f"\nPolicy {i}:")
    print(policy)

## Part 5: Integrated Waste Management Assistant

In this section, you will integrate all three models into a unified waste management assistant.

### 5.1 Design Integration Architecture

In [ ]:
# TODO: Design an architecture that integrates all three models
# - Create interfaces between components
# - Handle input/output flow

def classify_waste_image(image_path):
    """
    Classifies a waste image using the trained CNN.

    Args:
        image_path (str): Path to waste image

    Returns:
        str: Predicted waste category
    """

    # Load and resize image
    image = load_img(
        image_path,
        target_size=(IMG_HEIGHT, IMG_WIDTH)
    )

    # Convert image to array
    image_array = img_to_array(image)

    # Normalize pixel values
    image_array = image_array / 255.0

    # Add batch dimension
    image_array = np.expand_dims(image_array, axis=0)

    # Predict category
    prediction = model.predict(image_array, verbose=0)

    # Get predicted category
    predicted_index = np.argmax(prediction[0])

    category = class_names[predicted_index]

    return category

In [ ]:
def ecosort_assistant(description=None, image_path=None):
    """
    Integrated EcoSort Waste Management Assistant.

    Args:
        description (str): Waste item description
        image_path (str): Path to waste image

    Returns:
        dict: Classification and recycling instructions
    """

    # Step 1: Determine input type
    if description is not None and image_path is not None:
        raise ValueError("Provide either a description or an image, not both.")

    if description is not None:

        # Classify text
        category = classify_waste_description(description)
        input_type = "Text"

    elif image_path is not None:

        # Classify image
        category = classify_waste_image(image_path)
        input_type = "Image"

    else:
        raise ValueError("Please provide a description or image.")

    # Step 2: Generate recycling instructions
    instructions, policies = generate_recycling_instructions(
        category
    )

    # Step 3: Organize results
    result = {
        "input_type": input_type,
        "waste_category": category,
        "recycling_instructions": instructions,
        "relevant_policies": policies
    }

    return result

In [ ]:
# Test using a waste description
result = ecosort_assistant(
    description="empty plastic water bottle"
)

print("Input Type:", result["input_type"])
print("Predicted Category:", result["waste_category"])

print("\nRecycling Instructions:")
print(result["recycling_instructions"])

print("\nRelevant Policies:")
for policy in result["relevant_policies"]:
    print("-", policy)

In [ ]:
# Select an image from your dataset
image_path = "./realwaste-main/RealWaste/Plastic/Plastic_1.jpg"

# Classify image and generate instructions
result = ecosort_assistant(
    image_path=image_path
)

print("Input Type:", result["input_type"])
print("Predicted Category:", result["waste_category"])

print("\nRecycling Instructions:")
print(result["recycling_instructions"])

print("\nRelevant Policies:")
for policy in result["relevant_policies"]:
    print("-", policy)

### 5.2 Implement Integrated Assistant

In [ ]:
# TODO: Implement the integrated waste management assistant

def waste_management_assistant(input_data, input_type="image"):
    """
    Integrated waste management assistant that processes either images or text descriptions
    and returns waste classification and recycling instructions.

    Args:
        input_data: Either an image file path/array or a text description
        input_type (str): Type of input - "image" or "text"

    Returns:
        dict: Dictionary containing waste category, confidence, and recycling instructions
    """
    # Your code here
    # Step 1: Classify input
    if input_type == "image":

        # Load image from file path or array
        if isinstance(input_data, (str, bytes)):
            image = load_img(
                input_data,
                target_size=(IMG_HEIGHT, IMG_WIDTH)
            )
            image_array = img_to_array(image)

        else:
            image_array = np.asarray(input_data)
            image_array = tf.image.resize(
                image_array,
                (IMG_HEIGHT, IMG_WIDTH)
            ).numpy()

        # Normalize image if pixel values are 0-255
        image_array = image_array.astype("float32")

        if image_array.max() > 1.0:
            image_array = image_array / 255.0

        # Add batch dimension
        image_array = np.expand_dims(image_array, axis=0)

        # Predict waste category
        prediction = model.predict(image_array, verbose=0)[0]

        predicted_index = np.argmax(prediction)

        category = class_names[predicted_index]
        confidence = float(prediction[predicted_index])

    elif input_type == "text":

        # Clean text description
        cleaned_text = clean_text(input_data)

        # Convert text to TF-IDF features
        features = tfidf.transform([cleaned_text])

        # Predict waste category probabilities
        probabilities = text_model.predict_proba(features)[0]

        predicted_index = np.argmax(probabilities)

        # Convert predicted label to category name
        predicted_label = text_model.classes_[predicted_index]

        category = label_encoder.inverse_transform(
            [predicted_label]
        )[0]

        confidence = float(probabilities[predicted_index])

    else:
        raise ValueError("input_type must be 'image' or 'text'")

    # Step 2: Generate recycling instructions using RAG
    instructions, policies = generate_recycling_instructions(
        category
    )

    # Step 3: Return combined results
    result = {
        "input_type": input_type,
        "waste_category": category,
        "confidence": round(confidence, 4),
        "recycling_instructions": instructions,
        "relevant_policies": policies
    }

    return result

### 5.3 Evaluate the Integrated System

In [ ]:
# TODO: Evaluate the integrated system on test cases
# - Test with images from test dataset
# - Test with text descriptions from test dataset
# - Assess overall performance

# Evaluate integrated assistant using test images

image_results = []

# Select one batch of test images
for images, labels in test_dataset.take(1):

    # Test the first 10 images
    for i in range(min(10, len(images))):

        image = images[i].numpy()

        # Get actual category
        actual_index = np.argmax(labels[i].numpy())
        actual_category = class_names[actual_index]

        # Run integrated assistant
        result = waste_management_assistant(
            image,
            input_type="image"
        )

        # Store results
        image_results.append({
            "Actual": actual_category,
            "Predicted": result["waste_category"],
            "Confidence": result["confidence"],
            "Instructions": result["recycling_instructions"]
        })

# Convert results to DataFrame
image_results_df = pd.DataFrame(image_results)

display(image_results_df)

In [ ]:
# Evaluate integrated assistant using test descriptions

text_results = []

# Select 10 test descriptions
sample_indices = X_test.sample(
    n=min(10, len(X_test)),
    random_state=42
).index

for idx in sample_indices:

    description = X_test.loc[idx]

    # Get actual category
    actual_label = y_test.loc[idx]
    actual_category = label_encoder.inverse_transform(
        [actual_label]
    )[0]

    # Run integrated assistant
    result = waste_management_assistant(
        description,
        input_type="text"
    )

    # Store results
    text_results.append({
        "Description": description,
        "Actual": actual_category,
        "Predicted": result["waste_category"],
        "Confidence": result["confidence"],
        "Instructions": result["recycling_instructions"]
    })

# Convert results to DataFrame
text_results_df = pd.DataFrame(text_results)

display(text_results_df)

In [ ]:
# Calculate image classification accuracy
image_accuracy = accuracy_score(
    image_results_df["Actual"],
    image_results_df["Predicted"]
)

# Calculate text classification accuracy
text_accuracy = accuracy_score(
    text_results_df["Actual"],
    text_results_df["Predicted"]
)

print("Integrated Assistant Classification Performance")
print("-" * 50)

print(f"Image Classification Accuracy: {image_accuracy * 100:.2f}%")
print(f"Text Classification Accuracy: {text_accuracy * 100:.2f}%")

In [ ]:
# Calculate average confidence
image_confidence = image_results_df["Confidence"].mean()
text_confidence = text_results_df["Confidence"].mean()

# Create performance summary
performance_df = pd.DataFrame({
    "Input Type": ["Image", "Text"],
    "Test Cases": [
        len(image_results_df),
        len(text_results_df)
    ],
    "Accuracy (%)": [
        round(image_accuracy * 100, 2),
        round(text_accuracy * 100, 2)
    ],
    "Average Confidence (%)": [
        round(image_confidence * 100, 2),
        round(text_confidence * 100, 2)
    ]
})

display(performance_df)

## Submission Guidelines

1. Make sure all code cells are properly commented and annotated
2. Ensure that all functions are implemented and working correctly
3. Verify that all evaluation metrics are calculated and analyzed
4. Double-check that the integrated system works as expected
5. Submit your completed and annotated Jupyter notebook file

Remember to demonstrate your understanding of the underlying concepts and provide justification for your design decisions throughout the notebook.